# 08 — Activity Frequency and Seasonality

These reports describe recorded task starts, not health or productivity. Changes may reflect seasons, travel, or recording habits. Filters use exact names; `None` includes all values. Choose a task type to focus rolling and seasonal comparisons on one activity. Missing records are not proof of inactivity. Engagement-domain reports (including brief section 5) are deferred with the section 3 schema enhancement.


In [ ]:
%run pathutils.ipynb
%run database.ipynb
%run export.ipynb
%run reporting-utils.ipynb

import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt

sys.path.insert(0, str(Path(get_project_root_folder()) / "reports"))
from support.activity_analysis import (
    ReportPeriod,
    active_days,
    filter_history,
    rolling_profile,
    yearly_comparison,
)

In [ ]:
# Inclusive dates; END_DATE is also the as-of date. Use finite ISO dates.
START_DATE = "2024-01-01"
END_DATE = date.today().isoformat()
CATEGORY = None
TASK_TYPE = None
LOCATION = None
ROLLING_WINDOW = 90
DORMANCY_THRESHOLD = 90

In [ ]:
period = ReportPeriod(START_DATE, END_DATE, CATEGORY, TASK_TYPE, LOCATION)
# Load earlier observations for rolling windows, continuity and the full backlog.
query = construct_query("task-history.sql", {"START-DATE": "0001-01-01", "END-DATE": END_DATE})
history = filter_history(query_data(query), period)
filters = f"category={CATEGORY or 'All'}, type={TASK_TYPE or 'All'}, location={LOCATION or 'All'}"
period_label = f"{START_DATE} to {END_DATE}\n{filters}"
print(period_label)

## Active days by task type

Multiple starts on one day count once per type. The denominator includes every calendar day in the selected period. Previously observed types with no starts receive zero.


In [ ]:
frequency = active_days(history, period)
frequency

In [ ]:
if not frequency.empty and len(frequency.columns) > 1:
    figure, axes = plt.subplots(
        figsize=(CHART_FIGURE_SIZE[0], max(CHART_FIGURE_SIZE[1], len(frequency) * 0.3))
    )
    frequency.assign(Activity=frequency["Category"] + " / " + frequency["Task Type"]).plot(
        x="Activity", y=["Active Days"], kind="barh", stacked=False, ax=axes
    )
    format_chart(
        figure,
        axes,
        f"Active days by task type — {period_label}",
        "Days",
        "Activity",
        grid_axis="x",
    )
    export_chart(get_export_folder_path(), "active-days-by-type", "png", figure)
    plt.show()
else:
    print("No observations available for this chart.")

## Rolling activity profile

Windows include the displayed day and the preceding window minus one days. Earlier history supplies context; unavailable earlier records cannot establish absence. The default is 90 days; use 30 for a shorter view.


In [ ]:
rolling = rolling_profile(history, period, ROLLING_WINDOW)
rolling

In [ ]:
if not rolling.empty and len(rolling.columns) > 1:
    figure, axes = plt.subplots(figsize=CHART_FIGURE_SIZE)
    rolling.plot(
        x="Date", y=["Active Days", "Distinct Task Types"], kind="line", stacked=False, ax=axes
    )
    format_chart(
        figure,
        axes,
        f"Rolling {ROLLING_WINDOW}-day activity — {period_label}",
        "Date",
        "Count",
        grid_axis="y",
    )
    export_chart(get_export_folder_path(), "rolling-activity", "png", figure)
    plt.show()
else:
    print("No observations available for this chart.")

## Year-on-year comparison

Compare the same month across years. Partial months are flagged in the table and omitted from the chart to avoid comparing incomplete and complete months. Active days count any selected activity once per date.


In [ ]:
seasonality = yearly_comparison(history, period)
seasonality

In [ ]:
seasonal_chart = (
    seasonality.loc[~seasonality["Partial Month"]]
    .pivot(index="Month", columns="Year", values="Active Days")
    .reindex(range(1, 13))
    .reset_index()
)
if not seasonal_chart.empty and len(seasonal_chart.columns) > 1:
    figure, axes = plt.subplots(figsize=CHART_FIGURE_SIZE)
    seasonal_chart.plot(x="Month", y=None, kind="line", stacked=False, ax=axes)
    format_chart(
        figure,
        axes,
        f"Active days by month and year — {period_label}",
        "Month",
        "Active days",
        grid_axis="y",
    )
    export_chart(get_export_folder_path(), "year-on-year", "png", figure)
    plt.show()
else:
    print("No observations available for this chart.")

## Export

Tables are exported as XLSX and charts as PNG to `data/reports/` (or the runner’s configured destination).


In [ ]:
export_to_spreadsheet(
    get_export_folder_path(),
    "activity-frequency-and-seasonality.xlsx",
    {"Active Days": frequency, "Rolling Profile": rolling, "Year on Year": seasonality},
)